In [1]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

GPU Available: True
Device Name: Quadro M1200


In [ ]:
import sys; print(sys.executable)

- In a terminal run Ollama first with ```ollama list``` or ```ollama run <model_name>```.
- *Exit the chat*: Type ```/bye``` inside the active chat session.

### OpenAI's Completions API Prompting

In [1]:
from openai import OpenAI

# No API key needed for local Ollama. The OpenAI-compatible endpoint ignores
# the key, but the SDK still requires the parameter to be a non-empty string.
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

In [2]:
prompt_input = """Write a concise message to remind
users to be vigilant about phishing attacks."""

response = client.chat.completions.create(
    model="llama3.2:1b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt_input},
    ],
)
print(response)

ChatCompletion(id='chatcmpl-527', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Here\'s a concise message you can send to remind users to be vigilant about phishing attacks:\n\n"Remember, be cautious of unsolicited emails or messages that ask for sensitive information, like login credentials or financial details. Phishing scams are designed to trick you into revealing confidential info. Always verify sender addresses and check for spelling mistakes before responding or taking action."', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))], created=1783242505, model='llama3.2:1b', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=71, prompt_tokens=46, total_tokens=117, completion_tokens_details=None, prompt_tokens_details=None))


In [3]:
print(response.choices[0].message.content)

Here's a concise message you can send to remind users to be vigilant about phishing attacks:

"Remember, be cautious of unsolicited emails or messages that ask for sensitive information, like login credentials or financial details. Phishing scams are designed to trick you into revealing confidential info. Always verify sender addresses and check for spelling mistakes before responding or taking action."


### Prompting with LangChain

In [5]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:1b")

In [5]:
prompt_input = """Write a concise message to remind
users to be vigilant about phishing attacks."""

response = llm.invoke(prompt_input)
print(response.content)

Here's a concise message you can send to users to remind them to be vigilant about phishing attacks:

"Important Security Reminder: Be cautious of suspicious emails and messages that ask for personal info or login credentials. Phishing attempts are common, but they're usually fake. Always verify the sender's identity and never click on links from unknown sources. If in doubt, contact our support team directly."


### LangChain Prompt Templates

In [6]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    """
    You are an experienced copywriter. 
    Write a {num_words} words summary the the following text, 
    using a {tone} tone: {text}
    """
)

In [7]:
some_text = """The Aqueduct of Segovia (Spanish: 
Acueducto de Segovia) is a Roman aqueduct in Segovia, Spain. 
It was built around the first century AD to channel water from 
springs in the mountains 17 kilometres (11 mi) away to the 
city's fountains, public baths and private houses, and was in 
use until 1973. 
Its elevated section, with its complete arcade of 167 arches, 
is one of the best-preserved Roman aqueduct bridges and the 
foremost symbol of Segovia, as evidenced by its presence on the 
city's coat of arms. 
The Old Town of Segovia and the aqueduct, were declared a UNESCO 
World Heritage Site in 1985. As the aqueduct lacks a legible 
inscription (one was apparently located in the structure's attic, 
or top portion[citation needed]), the date of construction cannot be 
definitively determined. The general date of the Aqueduct's 
construction was long a mystery, although it was thought to have 
been during the 1st century AD, during the reigns of the Emperors 
Domitian, Nerva, and Trajan. At the end of the 20th century, 
Géza Alföldy deciphered the text on the dedication plaque by 
studying the anchors that held the now missing bronze letters 
in place. He determined that Emperor Domitian (AD 81–96) ordered 
its construction[1] and the year 98 AD was proposed as the most 
likely date of completion.[2] However, in 2016 archeological 
evidence was published which points to a slightly later date, 
after 112 AD, during the government of Trajan or in the 
beginning of the government of emperor Hadrian, 
from 117 AD."""

In [8]:
prompt = prompt_template.format(
    text=some_text,
    num_words=20,
    tone="knowledgable and engaging"
)

In [9]:
response = llm.invoke(prompt)
print(response.content)

"Discover the ancient Roman Aqueduct of Segovia, an engineering marvel that still thrives today."


### Langchain's FewShotPromptTemplate

LangChain lets us separate the training examples from the prompt template and inject them later.

In [10]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

examples = [
  {
      "number": 6,
      "reasoning": "not divisible by 5 nor by 7",
      "result": "None"
  },
  {
      "number": 15,
      "reasoning": "divisible by 5 but not by 7",
      "result": "Abra"
  },
  {
      "number": 12,
      "reasoning": "not divisible by 5 nor by 7",
      "result": "None"
  },
  {
      "number": 21,
      "reasoning": "divisible by 7 but not by 5",
      "result": "Kadabra"
  },
  {
      "number": 70,
      "reasoning": "divisible by 5 and by 7",
      "result": "Abra Kadabra"
  } ]

example_prompt = PromptTemplate(input_variables=["number", "reasoning", "result"], template="{number} \\ {reasoning} \\ {result}")
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Classify the following numbers as Abra, Kadabra or Abra Kadabra: {comma_delimited_input_numbers}",
    input_variables=["comma_delimited_input_numbers"]
)

prompt_input = few_shot_prompt.format(comma_delimited_input_numbers="3, 4, 5, 7, 8, 10, 11, 13, 35.")
response = llm.invoke(prompt_input)
print(response.content)

Let's classify each number according to its divisibility by 5 and/or 7:

- 3 not divisible by 5 nor by 7, so Abra
- 4 divisible by 5 but not by 7, so Kadabra
- 5 not divisible by 5 nor by 7, so Abra
- 7 divisible by 7 but not by 5, so Abra Kadabra (Wait, no!)
- 8 not divisible by 5 nor by 7, so None
- 10 divisible by 5 and by 7, so Abra Kadabra
- 11 not divisible by 5 nor by 7, so Abra
- 13 not divisible by 5 nor by 7, so None
- 35 divisible by 5 but not by 7, so Kadabra
